In [1]:
# If you're on a clean environment (Colab / new venv), run this once:
# !pip -q install biopython requests

In [2]:
from __future__ import annotations

import csv
import json
import logging
import re
import shutil
import tarfile
import time
import zipfile
from pathlib import Path
from typing import Dict, List, Optional, Set, Tuple

import requests
from Bio import Entrez

# =========================
# GLOBAL CONFIG (NCBI/PMC)
# =========================

# Official PMC Open Access endpoint: returns OA package links (.zip / .tar.gz)
OA_URL = "https://www.ncbi.nlm.nih.gov/pmc/utils/oa/oa.fcgi"

# Network tuning
REQUEST_TIMEOUT = 60
CHUNK_SIZE = 1024 * 256  # 256 KB chunks

# Entrez batching
ENTREZ_SUMMARY_BATCH = 100

# Logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("ncbi_pipeline")

# =========================
# FILTERING / SEARCH CONFIG
# =========================

MIN_PUBLICATION_YEAR = 2015

# Debug option: limit PMIDs per disease (None = no limit)
MAX_PMIDS_PER_DISEASE: Optional[int] = None

# "Executable / model" files -> data/
EXECUTABLE_EXTENSIONS = {
    ".py", ".r", ".m", ".ipynb",
    ".sbml", ".sedml", ".omex",
    ".cps", ".cellml", ".jl", ".cpp", ".c"
}

# "Support data" files (your current version: kept as data)
SUPPORT_DATA_EXTENSIONS = {
    ".csv", ".tsv", ".tab", ".txt", ".xls", ".xlsx", ".xlsm", ".ods",
    ".mat", ".h5", ".hdf5", ".sqlite", ".db",
    ".fasta", ".fa", ".fna", ".fastq", ".fq",
    ".gb", ".gbk", ".gff", ".gff3", ".bed", ".vcf",
    ".json", ".yaml", ".yml", ".graphml", ".gml", ".sif", ".xgmml",
    ".nwk", ".newick"
}

# Keywords to bias PubMed search toward "attached material" / code / archives
DEFAULT_KEYWORDS = [
    '"supplementary material"',
    '"supplementary data"',
    '"source code"',
    "python",
    '"R script"',
    "matlab",
    "notebook",
    "github",
    "gitlab",
    "bitbucket",
    "SBML",
    "OMEX",
    '"COMBINE archive"',
]

# Disease queries (edit freely)
DISEASE_QUERIES = {
    "dengue_files": '(dengue OR DENV)',
    "chikungunya_files": '(chikungunya OR CHIKV)',
    "lyme_files": '(lyme OR borrelia OR borreliosis)',
    "mpox_files": '(mpox OR monkeypox)',
    "west_nile_files": '("west nile" OR WNV)',
    "influenza_files": '(influenza OR "influenza virus" OR "avian influenza" OR H5N1)',
    "tuberculosis_files": '(tuberculosis OR TB OR mycobacterium)',
    "hiv_files": '(HIV OR "human immunodeficiency virus")',
    "covid_files": '("covid" OR "SARS-CoV-2")',
}

# Typical metadata files -> metadata/
METADATA_EXTENSIONS = {
    ".pdf", ".png", ".jpg", ".jpeg", ".svg", ".gif", ".tif", ".tiff",
    ".md", ".rst", ".bib", ".ris", ".nfo", ".doc", ".docx", ".ppt", ".pptx"
}

# Filename hints (often metadata)
METADATA_NAME_HINTS = {
    "readme", "license", "licence", "authors", "author", "manifest",
    "metadata", "supplement", "description", "legend", "notes", "protocol"
}

# Simple URL extractor to detect repo links inside text-ish files
URL_REGEX = re.compile(r'https?://[^\s"<>\]\)]+', flags=re.IGNORECASE)

In [3]:
def ensure_dir(path: Path) -> Path:
    """Create directory if missing (including parents)."""
    path.mkdir(parents=True, exist_ok=True)
    return path


def get_desktop_output_root() -> Path:
    """
    Output root on Desktop (Linux/FR can be ~/Bureau).
    Falls back to home if neither exists.
    """
    home = Path.home()
    desktop = home / "Desktop"
    bureau = home / "Bureau"

    if desktop.exists():
        root = desktop / "downloaded_ncbi_models"
    elif bureau.exists():
        root = bureau / "downloaded_ncbi_models"
    else:
        root = home / "downloaded_ncbi_models"

    return ensure_dir(root)


def normalize_oa_url(url: str) -> str:
    """PMC sometimes returns ftp:// links -> convert to https:// for requests."""
    return url.replace("ftp://", "https://", 1) if url.startswith("ftp://") else url


def safe_read_head(path: Path, n: int = 4000) -> bytes:
    """Read first N bytes safely (binary)."""
    try:
        return path.read_bytes()[:n]
    except Exception:
        return b""


def safe_read_text(path: Path, max_bytes: int = 200_000) -> str:
    """Read up to max_bytes as UTF-8-ish text (ignore errors)."""
    try:
        raw = path.read_bytes()[:max_bytes]
        return raw.decode("utf-8", errors="ignore")
    except Exception:
        return ""


def is_sbml_xml(path: Path) -> bool:
    """Heuristic: SBML XML contains '<sbml' near the top."""
    head = safe_read_head(path).lower()
    return b"<sbml" in head

In [4]:
def build_pubmed_query(disease_term: str, keywords: Optional[List[str]] = None) -> str:
    """Build a PubMed query with a publication-year filter."""
    if keywords is None:
        keywords = DEFAULT_KEYWORDS

    date_filter = f'("{MIN_PUBLICATION_YEAR}/01/01"[Date - Publication] : "3000"[Date - Publication])'
    return f"({disease_term}) AND ({' OR '.join(keywords)}) AND {date_filter}"


def is_archive_file(path: Path) -> bool:
    """Detect supported archive formats."""
    name = path.name.lower()
    return name.endswith(".zip") or name.endswith(".tar.gz") or name.endswith(".tgz")


def classify_file(path: Path) -> str:
    """
    Decide where a file goes:
    - 'data' for executables/models + SBML XML + support data (your current logic)
    - 'metadata' for typical metadata files
    - 'ignore' otherwise
    """
    suffix = path.suffix.lower()
    lower_name = path.name.lower()

    # Executables/models -> data
    if suffix in EXECUTABLE_EXTENSIONS:
        return "data"

    # XML: only SBML XML -> data, other XML -> metadata
    if suffix == ".xml":
        return "data" if is_sbml_xml(path) else "metadata"

    # Support data -> data
    if suffix in SUPPORT_DATA_EXTENSIONS:
        return "data"

    # Typical metadata -> metadata
    if suffix in METADATA_EXTENSIONS:
        return "metadata"

    # Filename hints -> metadata
    if any(h in lower_name for h in METADATA_NAME_HINTS):
        return "metadata"

    return "ignore"


def file_category_label(path: Path) -> str:
    """Human-readable category used in manifests."""
    suffix = path.suffix.lower()

    if suffix == ".xml":
        return "sbml_xml" if is_sbml_xml(path) else "xml_other"

    mapping = {
        ".py": "python",
        ".r": "r",
        ".m": "matlab",
        ".ipynb": "notebook",
        ".sbml": "sbml",
        ".sedml": "sedml",
        ".omex": "omex",
        ".cps": "cps",
        ".cellml": "cellml",
        ".jl": "julia",
        ".cpp": "cpp",
        ".c": "c",
        ".csv": "csv",
        ".tsv": "tsv",
        ".tab": "tabular",
        ".txt": "text_or_table",
        ".xls": "excel",
        ".xlsx": "excel",
        ".xlsm": "excel_macro",
        ".ods": "ods",
        ".json": "json",
        ".yaml": "yaml",
        ".yml": "yaml",
        ".sqlite": "sqlite",
        ".db": "database",
        ".mat": "matlab_binary",
        ".h5": "hdf5",
        ".hdf5": "hdf5",
        ".fasta": "fasta",
        ".fa": "fasta",
        ".fna": "fasta_nucleotide",
        ".fastq": "fastq",
        ".fq": "fastq",
        ".gb": "genbank",
        ".gbk": "genbank",
        ".gff": "gff",
        ".gff3": "gff3",
        ".bed": "bed",
        ".vcf": "vcf",
        ".graphml": "graphml",
        ".gml": "gml",
        ".sif": "sif",
        ".xgmml": "xgmml",
        ".nwk": "newick",
        ".newick": "newick",
        ".pdf": "pdf",
        ".md": "markdown",
    }
    return mapping.get(suffix, suffix[1:] if suffix else "other")

In [5]:
def has_minimum_useful_content(root_dir: Path, extra_links: Optional[List[str]] = None) -> bool:
    """
    Keep the article folder if we find at least one of:
    - any executable/model OR SBML XML
    - any repo link (github/gitlab/bitbucket/doi)
    - >= 2 support data files
    """
    strong_found = False
    support_count = 0

    for path in root_dir.rglob("*"):
        if not path.is_file():
            continue

        suffix = path.suffix.lower()

        if suffix in EXECUTABLE_EXTENSIONS:
            strong_found = True
            break

        if suffix == ".xml" and is_sbml_xml(path):
            strong_found = True
            break

        if suffix in SUPPORT_DATA_EXTENSIONS:
            support_count += 1

    if strong_found:
        return True

    if extra_links:
        if any(
            ("github.com" in x.lower()) or
            ("raw.githubusercontent.com" in x.lower()) or
            ("github.io" in x.lower()) or
            ("gitlab.com" in x.lower()) or
            ("bitbucket.org" in x.lower()) or
            ("doi.org" in x.lower())
            for x in extra_links
        ):
            return True

    return support_count >= 2


def save_csv(rows: List[Dict], path: Path) -> None:
    """Save rows to CSV (auto fieldnames)."""
    if not rows:
        return

    all_keys = set()
    for row in rows:
        all_keys.update(row.keys())

    preferred = ["pmcid", "pmid", "filename", "relative_path", "category", "source", "url", "title", "pubdate"]
    fieldnames = [k for k in preferred if k in all_keys] + sorted(all_keys - set(preferred))

    with open(path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fieldnames, extrasaction="ignore")
        w.writeheader()
        w.writerows(rows)


def save_json(data: dict, path: Path) -> None:
    """Save dict as pretty JSON."""
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def summarize_by_category(rows: List[Dict]) -> Dict[str, int]:
    """Count files per 'category' field."""
    counts: Dict[str, int] = {}
    for r in rows:
        cat = r["category"]
        counts[cat] = counts.get(cat, 0) + 1
    return counts

In [6]:
def extract_repo_links_from_text(text: str) -> List[str]:
    """Extract GitHub/GitLab/Bitbucket/DOI links from a text blob."""
    urls = URL_REGEX.findall(text or "")
    kept = []
    for u in urls:
        low = u.lower()
        if (
            "github.com" in low or
            "raw.githubusercontent.com" in low or
            "github.io" in low or
            "gitlab.com" in low or
            "bitbucket.org" in low or
            "doi.org" in low
        ):
            kept.append(u.rstrip(".,);]}>"))
    return sorted(set(kept))


def extract_repo_links_from_pubmed_records(records: List[Dict]) -> List[str]:
    """Extract repo links from PubMed summary fields."""
    links = set()
    for rec in records:
        for value in rec.values():
            if isinstance(value, str):
                for link in extract_repo_links_from_text(value):
                    links.add(link)
    return sorted(links)


def extract_repo_links_from_directory(root: Path) -> List[str]:
    """Scan extracted files (text-ish) for repo links."""
    links = set()
    text_like_extensions = {
        ".txt", ".md", ".rst", ".json", ".yaml", ".yml",
        ".py", ".r", ".m", ".ipynb", ".xml", ".html", ".htm", ".nxml"
    }

    for path in root.rglob("*"):
        if not path.is_file():
            continue
        if path.suffix.lower() not in text_like_extensions and path.suffix != "":
            continue

        text = safe_read_text(path)
        for link in extract_repo_links_from_text(text):
            links.add(link)

    return sorted(links)

In [7]:
def get_pmc_oa_links(pmcid: str) -> List[str]:
    """Get OA package links for a PMCID."""
    try:
        resp = requests.get(OA_URL, params={"id": pmcid}, timeout=REQUEST_TIMEOUT)
        resp.raise_for_status()
        hrefs = re.findall(r'href="([^"]+)"', resp.text)

        # Keep only archive links (.zip / .tar.gz / .tgz)
        return [
            normalize_oa_url(h)
            for h in hrefs
            if h.lower().endswith(".zip") or h.lower().endswith(".tar.gz") or h.lower().endswith(".tgz")
        ]
    except Exception as e:
        logger.warning("get_pmc_oa_links failed for %s: %s", pmcid, e)
        return []


def search_all_pubmed_ids(query: str, batch_size: int = 200) -> List[str]:
    """Paginated PubMed search: returns PMID list."""
    pmids: List[str] = []
    retstart = 0
    total_count = None

    while True:
        current_retmax = batch_size

        if MAX_PMIDS_PER_DISEASE is not None:
            remaining = MAX_PMIDS_PER_DISEASE - len(pmids)
            if remaining <= 0:
                break
            current_retmax = min(current_retmax, remaining)

        with Entrez.esearch(db="pubmed", term=query, retstart=retstart, retmax=current_retmax) as handle:
            result = Entrez.read(handle)

        if total_count is None:
            total_count = int(result.get("Count", 0))
            logger.info("PubMed total: %s", total_count)

        batch = result.get("IdList", [])
        if not batch:
            break

        pmids.extend(batch)
        retstart += len(batch)

        logger.info("PMIDs: %s / %s", len(pmids), total_count)

        if len(pmids) >= total_count:
            break

        time.sleep(0.34)  # avoid 429

    return pmids


def fetch_pubmed_summaries(pmids: List[str]) -> List[Dict]:
    """Fetch PubMed summaries (esummary) for PMIDs."""
    summaries: List[Dict] = []

    for i in range(0, len(pmids), ENTREZ_SUMMARY_BATCH):
        chunk = pmids[i:i + ENTREZ_SUMMARY_BATCH]

        with Entrez.esummary(db="pubmed", id=",".join(chunk), retmode="xml") as handle:
            result = Entrez.read(handle)

        docs = result if isinstance(result, list) else result.get("DocumentSummarySet", {}).get("DocumentSummary", [])
        for doc in docs:
            try:
                summaries.append(dict(doc))
            except Exception:
                summaries.append({"uid": str(doc)})

        logger.info("Summaries: %s / %s", min(i + ENTREZ_SUMMARY_BATCH, len(pmids)), len(pmids))
        time.sleep(0.34)

    return summaries


def pubmed_to_pmc(pmids: List[str]) -> Tuple[List[str], Dict[str, str]]:
    """Map PubMed IDs to PMC IDs using elink."""
    if not pmids:
        return [], {}

    pmid_to_pmcid: Dict[str, str] = {}
    pmcids = set()

    for i in range(0, len(pmids), ENTREZ_SUMMARY_BATCH):
        chunk = pmids[i:i + ENTREZ_SUMMARY_BATCH]

        with Entrez.elink(dbfrom="pubmed", db="pmc", id=",".join(chunk), linkname="pubmed_pmc") as handle:
            results = Entrez.read(handle)

        for linkset in results:
            pmid_list = linkset.get("IdList", [])
            source_pmid = str(pmid_list[0]) if pmid_list else None

            for db in linkset.get("LinkSetDb", []):
                for item in db.get("Link", []):
                    pmcid = f"PMC{item['Id']}"
                    pmcids.add(pmcid)
                    if source_pmid:
                        pmid_to_pmcid[source_pmid] = pmcid

        logger.info("PMID→PMCID mapping: %s / %s", min(i + ENTREZ_SUMMARY_BATCH, len(pmids)), len(pmids))
        time.sleep(0.34)

    return sorted(pmcids), pmid_to_pmcid

In [8]:
def recursively_extract_nested_archives(root_dir: Path, max_passes: int = 3) -> None:
    """
    Some OA packages contain nested zip/tar inside.
    This tries a few passes to unpack them.
    """
    seen: Set[Path] = set()

    for _ in range(max_passes):
        found_new = False

        for path in root_dir.rglob("*"):
            if not path.is_file():
                continue
            if path in seen:
                continue
            if not is_archive_file(path):
                continue

            lower = path.name.lower()
            target = path.parent / f"{path.stem}_extracted"
            if lower.endswith(".tar.gz"):
                target = path.parent / f"{path.name[:-7]}_extracted"
            elif lower.endswith(".tgz"):
                target = path.parent / f"{path.name[:-4]}_extracted"

            try:
                if lower.endswith(".zip"):
                    with zipfile.ZipFile(path, "r") as z:
                        z.extractall(target)
                else:
                    with tarfile.open(path, "r:gz") as t:
                        # filter="data" avoids some tar path issues on newer Python
                        t.extractall(target, filter="data")
                seen.add(path)
                found_new = True
            except Exception as e:
                logger.warning("Nested extraction failed for %s: %s", path, e)

        if not found_new:
            break


def collect_and_copy_files(
    source_root: Path,
    pmcid: str,
    data_dir: Path,
    metadata_dir: Path
) -> Tuple[List[Dict], List[Dict]]:
    """
    Copy files into data/ or metadata/ preserving the relative path from source_root.
    """
    d_rows: List[Dict] = []
    m_rows: List[Dict] = []
    copied_data: Set[Path] = set()
    copied_meta: Set[Path] = set()

    for path in source_root.rglob("*"):
        if not path.is_file():
            continue
        if is_archive_file(path):
            continue

        kind = classify_file(path)
        if kind == "ignore":
            continue

        rel_path = path.relative_to(source_root)
        row = {
            "pmcid": pmcid,
            "filename": path.name,
            "relative_path": str(rel_path),
            "category": file_category_label(path),
            "source": str(path),
        }

        if kind == "data":
            dest = data_dir / rel_path
            if dest not in copied_data:
                ensure_dir(dest.parent)
                shutil.copy2(path, dest)
                copied_data.add(dest)
            d_rows.append(row)

        else:
            dest = metadata_dir / rel_path
            if dest not in copied_meta:
                ensure_dir(dest.parent)
                shutil.copy2(path, dest)
                copied_meta.add(dest)
            m_rows.append(row)

    return d_rows, m_rows

In [9]:
def process_pmc_article(pmcid: str, disease_dir: Path, records: Dict, mapping: Dict) -> Dict:
    """
    For a given PMCID:
    - download OA package(s)
    - extract archives (including nested)
    - collect files into data/ and metadata/
    - write manifests + repository_links.csv
    """
    pmc_dir = ensure_dir(disease_dir / pmcid)
    tmp_dir = ensure_dir(pmc_dir / "_tmp_raw")
    data_dir = ensure_dir(pmc_dir / "data")
    metadata_dir = ensure_dir(pmc_dir / "metadata")

    oa_links = get_pmc_oa_links(pmcid)

    linked_records = [records[pmid] for pmid, linked in mapping.items() if linked == pmcid and pmid in records]
    repo_links_from_records = extract_repo_links_from_pubmed_records(linked_records)

    report = {
        "pmcid": pmcid,
        "status": "no_archive_found",
        "kept": False,
        "repo_links": repo_links_from_records,
    }

    # Save linked PubMed records info (nice for traceability)
    if linked_records:
        linked_rows = []
        for rec in linked_records:
            linked_rows.append({
                "pmcid": pmcid,
                "pmid": str(rec.get("uid", rec.get("Id", ""))),
                "title": str(rec.get("Title", "")),
                "pubdate": str(rec.get("PubDate", "")),
                "source": str(rec.get("Source", "")),
            })
        save_csv(linked_rows, metadata_dir / "pubmed_linked_records.csv")

    # No OA archives -> drop
    if not oa_links:
        shutil.rmtree(pmc_dir, ignore_errors=True)
        return report

    # Download + extract each OA archive
    any_extracted = False
    for i, link in enumerate(oa_links, start=1):
        try:
            archive_path = tmp_dir / Path(link).name
            extract_dir = tmp_dir / f"archive_{i}"

            with requests.get(link, stream=True, timeout=REQUEST_TIMEOUT) as r:
                r.raise_for_status()
                with open(archive_path, "wb") as f:
                    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)

            lower = archive_path.name.lower()
            if lower.endswith(".zip"):
                with zipfile.ZipFile(archive_path, "r") as z:
                    z.extractall(extract_dir)
            else:
                with tarfile.open(archive_path, "r:gz") as t:
                    t.extractall(extract_dir, filter="data")

            any_extracted = True

        except Exception as e:
            logger.warning("Extraction failed for %s: %s", pmcid, e)

    if not any_extracted:
        shutil.rmtree(pmc_dir, ignore_errors=True)
        return {"pmcid": pmcid, "status": "extract_failed", "kept": False}

    # Nested archives
    recursively_extract_nested_archives(tmp_dir)

    # Repo links inside extracted files
    repo_links_from_files = extract_repo_links_from_directory(tmp_dir)
    all_repo_links = sorted(set(repo_links_from_records + repo_links_from_files))

    # Keep / drop based on your rule
    if not has_minimum_useful_content(tmp_dir, extra_links=all_repo_links):
        shutil.rmtree(pmc_dir, ignore_errors=True)
        return {"pmcid": pmcid, "status": "no_useful_content", "kept": False, "repo_links": all_repo_links}

    # Copy files + manifests
    data_rows, metadata_rows = collect_and_copy_files(tmp_dir, pmcid, data_dir, metadata_dir)

    save_csv(data_rows, data_dir / "data_manifest.csv")
    save_json(
        {"pmcid": pmcid, "repo_links": all_repo_links, "counts": summarize_by_category(data_rows), "files": data_rows},
        data_dir / "data_manifest.json",
    )

    save_csv(metadata_rows, metadata_dir / "metadata_manifest.csv")
    save_json(
        {"pmcid": pmcid, "repo_links": all_repo_links, "counts": summarize_by_category(metadata_rows), "files": metadata_rows},
        metadata_dir / "metadata_manifest.json",
    )

    # Repository links CSV (the one you liked)
    if all_repo_links:
        repo_rows = [{"pmcid": pmcid, "url": u} for u in all_repo_links]
        save_csv(repo_rows, metadata_dir / "repository_links.csv")

    # Remove raw extraction directory (optional, keeps repo clean)
    shutil.rmtree(tmp_dir, ignore_errors=True)

    return {
        "pmcid": pmcid,
        "status": "ok",
        "kept": True,
        "repo_links": all_repo_links,
        "data_counts": summarize_by_category(data_rows),
        "metadata_counts": summarize_by_category(metadata_rows),
    }

In [10]:
def fetch_ncbi_associated_data(disease_term: str, out_dir: str, email: str) -> Dict:
    """
    Full disease pipeline:
    - PubMed search -> PMIDs
    - summaries (esummary)
    - map PMIDs -> PMCIDs
    - process each PMCID
    """
    Entrez.email = email

    base_dir = ensure_dir(Path(out_dir))

    query = build_pubmed_query(disease_term)
    logger.info("Query: %s", query)

    pmids = search_all_pubmed_ids(query)
    logger.info("PMIDs retrieved for %s: %d", out_dir, len(pmids))
    if not pmids:
        return {}

    summaries = fetch_pubmed_summaries(pmids)

    records: Dict[str, Dict] = {}
    for s in summaries:
        rid = s.get("Id") or s.get("uid")
        if rid:
            records[str(rid)] = s

    # Save PubMed summaries
    pubmed_rows = []
    for s in summaries:
        pubmed_rows.append({
            "pmid": str(s.get("uid", s.get("Id", ""))),
            "title": str(s.get("Title", "")),
            "pubdate": str(s.get("PubDate", "")),
            "source": str(s.get("Source", "")),
        })
    save_csv(pubmed_rows, base_dir / "pubmed_records.csv")
    save_json({"query": query, "records": summaries}, base_dir / "pubmed_records.json")

    # Map to PMC
    pmcids, pmid_to_pmcid = pubmed_to_pmc(pmids)
    logger.info("PMCIDs mapped for %s: %d", out_dir, len(pmcids))

    # Save mapping
    mapping_rows = [{"pmid": pmid, "pmcid": pmid_to_pmcid.get(pmid, "")} for pmid in pmids]
    save_csv(mapping_rows, base_dir / "pmc_mapping.csv")
    save_json({"mapping": pmid_to_pmcid, "pmcids": pmcids}, base_dir / "pmc_mapping.json")

    # Process each PMCID
    reports = []
    for pmcid in pmcids:
        reports.append(process_pmc_article(pmcid, base_dir, records, pmid_to_pmcid))

    save_json(reports, base_dir / "disease_summary.json")

    # disease_summary.csv (simple overview)
    summary_rows = []
    for r in reports:
        row = {
            "pmcid": r.get("pmcid", ""),
            "status": r.get("status", ""),
            "kept": r.get("kept", False),
            "repo_links_count": len(r.get("repo_links", [])),
        }
        for k, v in r.get("data_counts", {}).items():
            row[f"data_{k}"] = v
        for k, v in r.get("metadata_counts", {}).items():
            row[f"metadata_{k}"] = v
        summary_rows.append(row)

    save_csv(summary_rows, base_dir / "disease_summary.csv")

    return {
        "disease_term": disease_term,
        "query": query,
        "pmids": pmids,
        "pmcids": pmcids,
        "reports": reports,
    }

In [11]:
def run_all_diseases(email: str):
    """
    Runs the pipeline for all diseases in DISEASE_QUERIES.
    Output is written under ~/Desktop/downloaded_ncbi_models (or ~/Bureau).
    """
    output_root = get_desktop_output_root()
    logger.info("Output root: %s", output_root)

    for name, query in DISEASE_QUERIES.items():
        logger.info("\n" + "=" * 40 + f"\nDISEASE: {name}\n" + "=" * 40)
        disease_out_dir = output_root / name
        fetch_ncbi_associated_data(query, str(disease_out_dir), email)

In [12]:
# Important: NCBI Entrez requires a valid email address .
# Fill it locally before running.
EMAIL = "sebbahaya03@gmail.com"  

run_all_diseases(email=EMAIL)

INFO | Output root: /home/sebbah/Bureau/downloaded_ncbi_models
INFO | 
DISEASE: dengue_files
INFO | Query: ((dengue OR DENV)) AND ("supplementary material" OR "supplementary data" OR "source code" OR python OR "R script" OR matlab OR notebook OR github OR gitlab OR bitbucket OR SBML OR OMEX OR "COMBINE archive") AND ("2015/01/01"[Date - Publication] : "3000"[Date - Publication])
INFO | PubMed total: 98
INFO | PMIDs: 98 / 98
INFO | PMIDs retrieved for /home/sebbah/Bureau/downloaded_ncbi_models/dengue_files: 98
INFO | Summaries: 98 / 98
INFO | PMID→PMCID mapping: 98 / 98
INFO | PMCIDs mapped for /home/sebbah/Bureau/downloaded_ncbi_models/dengue_files: 84
INFO | 
DISEASE: chikungunya_files
INFO | Query: ((chikungunya OR CHIKV)) AND ("supplementary material" OR "supplementary data" OR "source code" OR python OR "R script" OR matlab OR notebook OR github OR gitlab OR bitbucket OR SBML OR OMEX OR "COMBINE archive") AND ("2015/01/01"[Date - Publication] : "3000"[Date - Publication])
INFO | Pu